
# Corrected Diebold-Li (2006) macro reconstruction notebook

This notebook rebuilds the three monthly macro series used alongside the Diebold-Li benchmark and
**corrects the one-month labeling error** found in the earlier version.

## Series
- **FFR**: Federal Funds Rate
- **CU**: Capacity Utilization: Manufacturing
- **PI**: year-over-year inflation from **PCEPI**

## Key correction
The benchmark comparison showed that the reconstructed panel matched almost perfectly under
`recon_t+1 vs bench_t`, which means the previous notebook was labeling each reconstructed value
**one month too late**.

This version fixes that by shifting the reconstructed **month label back by one month** while
keeping the underlying source observations unchanged.


In [1]:

# =========================
# 1. Imports and settings
# =========================
import os
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Set your API key here or use an environment variable
API_KEY = "8646e4c441bf17d906a058f8eddaf3f8"

if not API_KEY:
    raise ValueError("Set FRED_API_KEY in your environment or assign API_KEY directly in this cell.")

FRED_OBS_URL = "https://api.stlouisfed.org/fred/series/observations"

# Start one year early so 12-month inflation is available at 1972-01
START_BENCH = "1971-01-01"
END_BENCH   = "2000-12-31"

START_FULL  = "1971-01-01"
END_FULL    = pd.Timestamp.today().strftime("%Y-%m-%d")

BENCHMARK_PATH = Path("Benchmark.txt")  # optional local file


In [2]:

# =========================
# 2. FRED helpers
# =========================
def _request_fred(params, max_retries=5, pause=0.75, verbose=False):
    last_response = None
    for attempt in range(max_retries):
        r = requests.get(FRED_OBS_URL, params=params, timeout=60)
        last_response = r
        if r.status_code == 200:
            return r
        if verbose:
            print(f"Attempt {attempt+1} failed with status {r.status_code}")
            try:
                print(r.json())
            except Exception:
                print(r.text[:1000])
        time.sleep(pause * (attempt + 1))
    last_response.raise_for_status()

def fetch_fred_series(series_id, observation_start, observation_end):
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": observation_start,
        "observation_end": observation_end,
        "sort_order": "asc",
    }
    r = _request_fred(params)
    obs = r.json()["observations"]
    df = pd.DataFrame(obs)

    if df.empty:
        return pd.DataFrame(columns=["date", series_id])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[series_id] = pd.to_numeric(df["value"], errors="coerce")
    return df[["date", series_id]].sort_values("date").reset_index(drop=True)



## Why the label shift matters

The benchmark values are interpreted as monthly values attached to month `t`.  
If you construct the panel using information observed on the first day of month `t+1`, then the
result belongs to **month `t`**, not `t+1`.

For standard monthly FRED history, the same practical issue shows up if you compare the raw
observation month directly against the benchmark table and discover that the best fit is
`recon_t+1 vs bench_t`. The fix is to shift the **panel month label** back by one month.


In [3]:

# =========================
# 3. Build corrected benchmark-style monthly macro panel
# =========================
def build_dns_macro_panel_corrected(start_date, end_date):
    ffr = fetch_fred_series("FEDFUNDS", start_date, end_date)
    cu = fetch_fred_series("CUMFNS", start_date, end_date)
    pcepi = fetch_fred_series("PCEPI", start_date, end_date)

    macro = (
        ffr.merge(cu, on="date", how="outer")
           .merge(pcepi, on="date", how="outer")
           .sort_values("date")
           .reset_index(drop=True)
    )

    # Construct inflation first on the source monthly timeline
    macro["PI"] = 100.0 * (macro["PCEPI"] / macro["PCEPI"].shift(12) - 1.0)

    # Optional alternate inflation diagnostics
    macro["PI_yoy_log"] = 100.0 * (np.log(macro["PCEPI"]) - np.log(macro["PCEPI"].shift(12)))
    macro["PI_mom_ann"] = 1200.0 * (macro["PCEPI"] / macro["PCEPI"].shift(1) - 1.0)
    macro["PI_mom_log_ann"] = 1200.0 * (np.log(macro["PCEPI"]) - np.log(macro["PCEPI"].shift(1)))

    # Rename into the benchmark's variable names
    macro = macro.rename(columns={
        "FEDFUNDS": "FFR",
        "CUMFNS": "CU"
    })

    # ------------------------------------------------------------------
    # CRITICAL FIX:
    # Shift the benchmark panel month label BACK by one month.
    #
    # Example:
    # a source row dated 1973-02-01 is labeled to benchmark month 1973-01
    # ------------------------------------------------------------------
    macro["month"] = (macro["date"] - pd.offsets.MonthBegin(1)).dt.to_period("M")

    return macro[[
        "date", "month", "FFR", "CU", "PCEPI",
        "PI", "PI_yoy_log", "PI_mom_ann", "PI_mom_log_ann"
    ]].copy()

macro_1972_2000 = build_dns_macro_panel_corrected(START_BENCH, END_BENCH)
macro_1972_present = build_dns_macro_panel_corrected(START_FULL, END_FULL)

# Trim the benchmark-era panel to 1972M01-2000M12
macro_1972_2000 = macro_1972_2000[
    (macro_1972_2000["month"] >= pd.Period("1972-01", freq="M")) &
    (macro_1972_2000["month"] <= pd.Period("2000-12", freq="M"))
].reset_index(drop=True)

# Full panel begins at 1972M01 because the 1971 rows only exist to support PI construction
macro_1972_present = macro_1972_present[
    macro_1972_present["month"] >= pd.Period("1972-01", freq="M")
].reset_index(drop=True)

print("Benchmark-era corrected panel:")
display(macro_1972_2000.head(12))

print("Full corrected panel:")
display(macro_1972_present.head(12))
display(macro_1972_present.tail(12))


Benchmark-era corrected panel:


,date,month,FFR,CU,PCEPI,PI,PI_yoy_log,PI_mom_ann,PI_mom_log_ann
0,1972-02-01,1972-01,3.30,81.8563,21.098,3.951518,3.875443,4.739472,4.730137
1,1972-03-01,1972-02,3.83,82.2062,21.128,3.756814,3.687965,1.706323,1.705111
2,1972-04-01,1972-03,4.17,82.8926,21.160,3.507313,3.447208,1.817493,1.816118
3,1972-05-01,1972-04,4.27,82.8225,21.207,3.307677,3.254151,2.665406,2.662451
4,1972-06-01,1972-05,4.46,82.8930,21.239,3.006935,2.962613,1.810723,1.809358
5,1972-07-01,1972-06,4.55,82.6441,21.315,3.065616,3.019565,4.293987,4.286323
6,1972-08-01,1972-07,4.81,83.5285,21.377,3.066390,3.020316,3.490500,3.485433
7,1972-09-01,1972-08,4.87,83.9714,21.473,3.374735,3.319041,5.388969,5.376905
8,1972-10-01,1972-09,5.05,84.9134,21.497,3.331090,3.276811,1.341219,1.340470
9,1972-11-01,1972-10,5.06,85.7175,21.561,3.424953,3.367607,3.572592,3.567284


Full corrected panel:


,date,month,FFR,CU,PCEPI,PI,PI_yoy_log,PI_mom_ann,PI_mom_log_ann
0,1972-02-01,1972-01,3.30,81.8563,21.098,3.951518,3.875443,4.739472,4.730137
1,1972-03-01,1972-02,3.83,82.2062,21.128,3.756814,3.687965,1.706323,1.705111
2,1972-04-01,1972-03,4.17,82.8926,21.160,3.507313,3.447208,1.817493,1.816118
3,1972-05-01,1972-04,4.27,82.8225,21.207,3.307677,3.254151,2.665406,2.662451
4,1972-06-01,1972-05,4.46,82.8930,21.239,3.006935,2.962613,1.810723,1.809358
5,1972-07-01,1972-06,4.55,82.6441,21.315,3.065616,3.019565,4.293987,4.286323
6,1972-08-01,1972-07,4.81,83.5285,21.377,3.066390,3.020316,3.490500,3.485433
7,1972-09-01,1972-08,4.87,83.9714,21.473,3.374735,3.319041,5.388969,5.376905
8,1972-10-01,1972-09,5.05,84.9134,21.497,3.331090,3.276811,1.341219,1.340470
9,1972-11-01,1972-10,5.06,85.7175,21.561,3.424953,3.367607,3.572592,3.567284


,date,month,FFR,CU,PCEPI,PI,PI_yoy_log,PI_mom_ann,PI_mom_log_ann
638,2025-04-01,2025-03,4.33,75.6285,126.150,2.277426,2.251880,1.991409,1.989758
639,2025-05-01,2025-04,4.33,75.4680,126.380,2.458086,2.428361,2.187872,2.185880
640,2025-06-01,2025-05,4.33,75.6385,126.743,2.593513,2.560452,3.446748,3.441807
641,2025-07-01,2025-06,4.33,75.9194,126.960,2.605547,2.572181,2.054551,2.052795
642,2025-08-01,2025-07,4.33,75.8608,127.293,2.747621,2.710551,3.147448,3.143328
643,2025-09-01,2025-08,4.22,75.7980,127.625,2.787442,2.749300,3.129787,3.125713
644,2025-10-01,2025-09,4.09,75.1411,127.871,2.712581,2.676442,2.313026,2.310800
645,2025-11-01,2025-10,3.88,75.0295,128.152,2.820190,2.781155,2.637033,2.634139
646,2025-12-01,2025-11,3.72,74.9116,128.576,2.878084,2.837445,3.970285,3.963732
647,2026-01-01,2025-12,3.64,75.3110,128.965,2.828963,2.789686,3.630538,3.625057


In [4]:

# =========================
# 4. Save corrected outputs
# =========================
macro_1972_2000_save = macro_1972_2000.copy()
macro_1972_present_save = macro_1972_present.copy()

macro_1972_2000_save["month"] = macro_1972_2000_save["month"].astype(str)
macro_1972_present_save["month"] = macro_1972_present_save["month"].astype(str)

macro_1972_2000_save.to_csv("dns_macro_benchmark_style_CORRECTED_1972_2000.csv", index=False)
macro_1972_present_save.to_csv("dns_macro_benchmark_style_CORRECTED_1972_present.csv", index=False)

print("Saved:")
print(" - dns_macro_benchmark_style_CORRECTED_1972_2000.csv")
print(" - dns_macro_benchmark_style_CORRECTED_1972_present.csv")


Saved:
 - dns_macro_benchmark_style_CORRECTED_1972_2000.csv
 - dns_macro_benchmark_style_CORRECTED_1972_present.csv



## Optional benchmark check

The next cells are optional. They let you verify that the same-month comparison is now the correct one,
instead of the old notebook's one-month-ahead alignment.


In [5]:

# =========================
# 5. Optional: load benchmark
# =========================
if BENCHMARK_PATH.exists():
    benchmark = pd.read_csv(BENCHMARK_PATH, sep=r"\s+")
    benchmark["month"] = pd.PeriodIndex(
        benchmark["obs"].astype(str).str.replace("M", "-", regex=False),
        freq="M"
    )
    benchmark = benchmark[["month", "CU", "PI", "FFR"]].copy()
    print("Loaded benchmark file.")
    display(benchmark.head())
else:
    benchmark = None
    print("Benchmark.txt not found. Skipping comparison cells.")


Loaded benchmark file.


,month,CU,PI,FFR
0,1972-01,81.3204,3.920467,3.29
1,1972-02,81.7608,3.734112,3.83
2,1972-03,82.4567,3.523413,4.17
3,1972-04,82.4523,3.336742,4.27
4,1972-05,82.5176,3.036715,4.46


In [6]:

# =========================
# 6. Optional: same-month comparison after correction
# =========================
def summarize_error(df, err_col):
    x = df[err_col].dropna()
    return pd.Series({
        "N": len(x),
        "MAE": x.abs().mean(),
        "RMSE": np.sqrt((x ** 2).mean()),
        "MaxAbsErr": x.abs().max(),
        "MeanErr": x.mean()
    })

if benchmark is not None:
    compare = benchmark.merge(
        macro_1972_2000[["month", "FFR", "CU", "PI"]],
        on="month",
        how="inner",
        suffixes=("_bench", "_recon")
    )

    compare["err_FFR"] = compare["FFR_recon"] - compare["FFR_bench"]
    compare["err_CU"] = compare["CU_recon"] - compare["CU_bench"]
    compare["err_PI"] = compare["PI_recon"] - compare["PI_bench"]

    summary = pd.DataFrame({
        "FFR": summarize_error(compare, "err_FFR"),
        "CU": summarize_error(compare, "err_CU"),
        "PI": summarize_error(compare, "err_PI"),
    }).T

    print("=== SAME-MONTH ERROR SUMMARY AFTER CORRECTION ===")
    display(summary)

    compare_save = compare.copy()
    compare_save["month"] = compare_save["month"].astype(str)
    compare_save.to_csv("dns_macro_same_month_vs_benchmark_CORRECTED.csv", index=False)
    print("Saved: dns_macro_same_month_vs_benchmark_CORRECTED.csv")
else:
    compare = None


=== SAME-MONTH ERROR SUMMARY AFTER CORRECTION ===


,N,MAE,RMSE,MaxAbsErr,MeanErr
FFR,347.0,0.000301,0.001406,0.01000,0.000006
CU,347.0,0.324485,0.432135,1.66170,0.123484
PI,347.0,0.232713,0.314238,0.89592,0.044480


Saved: dns_macro_same_month_vs_benchmark_CORRECTED.csv


In [7]:

# =========================
# 7. Final display
# =========================
print("Corrected benchmark-style 1972-2000 panel:")
display(macro_1972_2000.head(15))

print("Corrected benchmark-style 1972-present panel:")
display(macro_1972_present.tail(15))


Corrected benchmark-style 1972-2000 panel:


,date,month,FFR,CU,PCEPI,PI,PI_yoy_log,PI_mom_ann,PI_mom_log_ann
0,1972-02-01,1972-01,3.30,81.8563,21.098,3.951518,3.875443,4.739472,4.730137
1,1972-03-01,1972-02,3.83,82.2062,21.128,3.756814,3.687965,1.706323,1.705111
2,1972-04-01,1972-03,4.17,82.8926,21.160,3.507313,3.447208,1.817493,1.816118
3,1972-05-01,1972-04,4.27,82.8225,21.207,3.307677,3.254151,2.665406,2.662451
4,1972-06-01,1972-05,4.46,82.8930,21.239,3.006935,2.962613,1.810723,1.809358
5,1972-07-01,1972-06,4.55,82.6441,21.315,3.065616,3.019565,4.293987,4.286323
6,1972-08-01,1972-07,4.81,83.5285,21.377,3.066390,3.020316,3.490500,3.485433
7,1972-09-01,1972-08,4.87,83.9714,21.473,3.374735,3.319041,5.388969,5.376905
8,1972-10-01,1972-09,5.05,84.9134,21.497,3.331090,3.276811,1.341219,1.340470
9,1972-11-01,1972-10,5.06,85.7175,21.561,3.424953,3.367607,3.572592,3.567284


Corrected benchmark-style 1972-present panel:


,date,month,FFR,CU,PCEPI,PI,PI_yoy_log,PI_mom_ann,PI_mom_log_ann
635,2025-01-01,2024-12,4.33,74.6327,125.417,2.607380,2.573967,4.205507,4.198154
636,2025-02-01,2025-01,4.33,75.5230,125.921,2.710485,2.674402,4.822313,4.812649
637,2025-03-01,2025-02,4.33,75.7850,125.941,2.359434,2.332029,0.190596,0.190581
638,2025-04-01,2025-03,4.33,75.6285,126.150,2.277426,2.251880,1.991409,1.989758
639,2025-05-01,2025-04,4.33,75.4680,126.380,2.458086,2.428361,2.187872,2.185880
640,2025-06-01,2025-05,4.33,75.6385,126.743,2.593513,2.560452,3.446748,3.441807
641,2025-07-01,2025-06,4.33,75.9194,126.960,2.605547,2.572181,2.054551,2.052795
642,2025-08-01,2025-07,4.33,75.8608,127.293,2.747621,2.710551,3.147448,3.143328
643,2025-09-01,2025-08,4.22,75.7980,127.625,2.787442,2.749300,3.129787,3.125713
644,2025-10-01,2025-09,4.09,75.1411,127.871,2.712581,2.676442,2.313026,2.310800
